# Inference on the Full Real-World Dataset

This notebook takes the trained **MobileNetV3-02** classifier (86.5% test accuracy, 0.83 macro-F1 — see `README.md`) and runs it on a new batch of REPSOL spectrograms that has **no ground-truth labels**. This is the "deploy the model" step (README `Next Steps` #5).

## What's different from everything you've done so far

Every notebook up to now (`Models/*.ipynb`, `Evaluation.ipynb`) measured the model against a **test set with known labels** — you could compute accuracy, F1, a confusion matrix, because you knew the right answer for every sample. That is *validation*.

This is *deployment*. The new data from your boss has no `category` column. There is no right answer to check against, so there is no accuracy number to report here — and that's expected, not a gap in this notebook.

## What the output actually looks like

For every input file you get, per file:
- a **predicted class** (0-7)
- a **confidence score** (the softmax probability of that class, 0-1)
- the **top-3 candidate classes** with their probabilities
- a **needs_review flag**: `True` if confidence is low, or if the predicted class is one of the two rare/hard classes (1 = Hammering, 7 = Works/sirens)

That's a CSV, not a verdict. The deliverable of this notebook is:
1. `outputs/inference/predictions_<timestamp>.csv` — every prediction
2. `outputs/inference/review_queue_<timestamp>.csv` — the subset a human should look at before anyone acts on the label
3. A couple of diagnostic charts to sanity-check the run *as a whole* before trusting any individual prediction

## How to judge whether the run "worked" (no ground truth available)

Since you can't compute accuracy, judge the run at the **batch level** instead of the sample level:

| Check | Good sign | Red flag |
|---|---|---|
| Predicted class mix vs. training base rates | Roughly similar shape (classes 3 & 6 common, 1 & 7 rare) | Wildly different, e.g. one class eating >80% of predictions |
| Confidence distribution | Most mass above ~0.7, a long thin tail below | Uniformly low confidence across the board |
| Review queue size | A manageable minority (rule of thumb: <25-30%) | Most of the batch flagged |
| Manual spot-check of ~20-30 predictions (below) | Predicted class looks plausible against the spectrogram | Predictions look arbitrary/random |

**If confidence is uniformly low or the class mix looks nonsensical, the first suspect is preprocessing parity, not the model.** This model is picky about its inputs (see the data pipeline fix in `README.md` — an off-by-one crop cost 13 accuracy points). The new data must go through the *identical* chain: same MATLAB render settings → crop to the plot interior → resize to 224×224 → z-score. This notebook reuses the exact same functions the training pipeline used (`src/preprocess/generate_spectrograms_from_images.py` + `src/preprocess/downsize_spectrograms.py`) rather than re-implementing them, specifically to avoid a second version of that bug.

## The review workflow this feeds into

Per the README's deployment guidance: **auto-accept** high-confidence predictions, **route to a human** everything below the confidence threshold and anything predicted as class 1 or 7 (too few training examples to trust blindly). A domain expert then confirms or corrects the flagged rows. Those corrections are valuable — they're new labeled data for classes 1 & 7, which is the single biggest lever left per the README ("More data for classes 1 & 7").

## 1. Config

Edit `NEW_DATA_DIR` to point at the folder your boss gives you. Everything else has a sensible default.

In [ ]:
from pathlib import Path
import sys
import time
import torch

# ===== Paths =====
PROJECT_ROOT = Path(r"D:\Work\Internships\INMAR\REPSOL")
sys.path.insert(0, str(PROJECT_ROOT))

# vvv EDIT THIS: folder containing the new, unlabeled spectrogram PNGs.
# Same rendering as training data: "*_spectrogram_win16384.png" (MATLAB, win=16384).
# Can be a flat folder or nested in subfolders — we search recursively.
NEW_DATA_DIR = Path(r"D:\REPSOL_Classification_NEW")

CHECKPOINT_PATH = PROJECT_ROOT / "Models_output" / "mobilenetv3_02_best_01.pth"  # the 🏆 best model
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "inference"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Class names — must match the sorted folder order the model was trained with =====
CLASS_NAMES = [
    "0 ActividadBASE_NO pattern activity",
    "1 Pulses HAMMERING",
    "2 Marked cycles 3 segundos",
    "3 Continuous activity & tone 3.15kHz_SPL ALTO",
    "4 Continuous activity & tone 3.15kHz_ SPL BAJO",
    "5 Blasts",
    "6 Machinery continuous activity",
    "7 Works_ sirens and knocks en altas frecuencias RAFAGAS a 3.15",
]
SHORT_NAMES = [
    "0 Baseline", "1 Hammering", "2 Marked cycles", "3 Tone ALTO",
    "4 Tone BAJO", "5 Blasts", "6 Machinery", "7 Works/sirens",
]
NUM_CLASSES = len(CLASS_NAMES)

# Training-set base rates (train+val+test combined, from README) — used later as a sanity-check reference
TRAIN_BASE_COUNTS = [225, 43, 132, 703, 255, 73, 503, 41]

# ===== Deployment thresholds (from README's deployment guidance) =====
CONFIDENCE_THRESHOLD = 0.70   # below this -> needs_review
RARE_CLASSES = [1, 7]         # always route to human review regardless of confidence

BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID = time.strftime("%Y%m%d_%H%M%S")

print("NEW_DATA_DIR   :", NEW_DATA_DIR, "(exists:", NEW_DATA_DIR.exists(), ")")
print("CHECKPOINT_PATH:", CHECKPOINT_PATH, "(exists:", CHECKPOINT_PATH.exists(), ")")
print("DEVICE         :", DEVICE)
print("RUN_ID         :", RUN_ID)

## 2. Preprocessing — reuse the training-time functions, don't re-implement them

`image_to_tensor` (PNG → z-scored RGB tensor) and `downsize_tensor` (crop plot interior → resize 224×224 → re-z-score) are imported straight from `src/preprocess/`. This guarantees byte-for-byte the same transform the model was trained on.

In [ ]:
from src.preprocess.generate_spectrograms_from_images import image_to_tensor
from src.preprocess.downsize_spectrograms import downsize_tensor, CROP_ROWS, CROP_COLS, TARGET_SIZE

print(f"Crop rows={CROP_ROWS} cols={CROP_COLS} -> resize to {TARGET_SIZE}")

# Sanity check: run the chain on one known TRAIN sample and confirm shape/scale look right.
_sample_pngs = list((PROJECT_ROOT / "Data").rglob("*.png"))
if _sample_pngs:
    _t = image_to_tensor(_sample_pngs[0])
    if _t is not None:
        _t = downsize_tensor(_t.float())
        print(f"Sample transform OK: shape={tuple(_t.shape)}  mean={_t.mean():.3f}  std={_t.std():.3f}")
        assert tuple(_t.shape) == (3, 224, 224), "Unexpected output shape from the preprocessing chain"
else:
    print("No local PNGs found under Data/ to sanity-check against — skipping (not required to proceed).")

## 3. Discover the new files

In [ ]:
assert NEW_DATA_DIR.exists(), f"NEW_DATA_DIR does not exist: {NEW_DATA_DIR} — point it at the folder your boss gave you."

png_files = sorted(NEW_DATA_DIR.rglob("*_spectrogram_win16384.png"))
if not png_files:
    print("No '*_spectrogram_win16384.png' files found — falling back to any '*.png' under NEW_DATA_DIR.")
    png_files = sorted(NEW_DATA_DIR.rglob("*.png"))

print(f"Found {len(png_files)} spectrogram images under {NEW_DATA_DIR}")
assert len(png_files) > 0, "No PNG files found at all — check NEW_DATA_DIR and the file naming."
for p in png_files[:5]:
    print(" ", p.relative_to(NEW_DATA_DIR))

## 4. Dataset & DataLoader

A thin `Dataset` that runs each PNG through the exact same transform chain as training and carries the source path alongside the tensor (so we can trace every prediction back to a file).

In [ ]:
from torch.utils.data import Dataset, DataLoader

class InferenceSpectrogramDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        tensor = image_to_tensor(path)
        if tensor is None:
            # Unreadable/corrupt file — return a placeholder, flagged as ok=False so it's
            # never silently scored as a real prediction.
            return torch.zeros(3, *TARGET_SIZE), str(path), False
        tensor = downsize_tensor(tensor.float())
        return tensor, str(path), True


def collate_inference(batch):
    tensors, paths, oks = zip(*batch)
    return torch.stack(tensors), list(paths), list(oks)


inference_ds = InferenceSpectrogramDataset(png_files)
inference_loader = DataLoader(
    inference_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, collate_fn=collate_inference,
)
print(f"{len(inference_ds)} files -> {len(inference_loader)} batches of up to {BATCH_SIZE}")

## 5. Load the model

MobileNetV3-02 is the README's pick for deployment: best accuracy, best hard-class balance, smallest/fastest. If you later want to compare against EfficientNet-04 (the ensemble partner), swap the checkpoint path and model class here.

In [ ]:
from src.MobileNet.model import MobileNetV3Spectrogram

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

model = MobileNetV3Spectrogram(num_classes=NUM_CLASSES, freeze_backbone=False).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {CHECKPOINT_PATH.name}  ({n_params/1e6:.1f}M params)  device={DEVICE}")

## 6. Run inference

For each file: softmax probabilities → top-3 classes → confidence (= max probability).

In [ ]:
import torch.nn.functional as F
import pandas as pd

rows = []
start = time.time()

with torch.no_grad():
    for batch_idx, (x, paths, oks) in enumerate(inference_loader, 1):
        x = x.to(DEVICE)
        probs = F.softmax(model(x), dim=1).cpu()
        top3_probs, top3_idx = probs.topk(3, dim=1)

        for i, (path, ok) in enumerate(zip(paths, oks)):
            if not ok:
                rows.append({
                    "filepath": path, "filename": Path(path).name,
                    "pred_class": None, "pred_name": None, "confidence": float("nan"),
                    "top1_class": None, "top1_prob": float("nan"),
                    "top2_class": None, "top2_prob": float("nan"),
                    "top3_class": None, "top3_prob": float("nan"),
                    "error": "unreadable_image",
                })
                continue
            pred_idx = int(top3_idx[i, 0])
            rows.append({
                "filepath": path, "filename": Path(path).name,
                "pred_class": pred_idx, "pred_name": CLASS_NAMES[pred_idx],
                "confidence": float(top3_probs[i, 0]),
                "top1_class": pred_idx, "top1_prob": float(top3_probs[i, 0]),
                "top2_class": int(top3_idx[i, 1]), "top2_prob": float(top3_probs[i, 1]),
                "top3_class": int(top3_idx[i, 2]), "top3_prob": float(top3_probs[i, 2]),
                "error": None,
            })

        if batch_idx % 10 == 0 or batch_idx == len(inference_loader):
            elapsed = time.time() - start
            print(f"  [{batch_idx}/{len(inference_loader)}] {elapsed:.1f}s elapsed", flush=True)

results = pd.DataFrame(rows)
print(f"\nDone: {len(results)} files, {results['error'].notna().sum()} unreadable, "
      f"in {time.time()-start:.1f}s")
results.head()

## 7. Flag for human review & save

Per the README's deployment guidance: auto-accept confident bulk-class predictions; send low-confidence and rare-class (1, 7) predictions — plus anything that failed to load — to a human.

In [ ]:
results["needs_review"] = (
    results["error"].notna()
    | (results["confidence"] < CONFIDENCE_THRESHOLD)
    | results["pred_class"].isin(RARE_CLASSES)
)
results = results.sort_values("confidence", ascending=True, na_position="first").reset_index(drop=True)

predictions_path = OUTPUT_DIR / f"predictions_{RUN_ID}.csv"
review_queue_path = OUTPUT_DIR / f"review_queue_{RUN_ID}.csv"

results.to_csv(predictions_path, index=False)
results[results["needs_review"]].to_csv(review_queue_path, index=False)

n_review = int(results["needs_review"].sum())
print(f"Saved all predictions   -> {predictions_path}")
print(f"Saved review queue      -> {review_queue_path}")
print(f"Review queue: {n_review}/{len(results)} ({100*n_review/len(results):.1f}%) flagged")

## 8. Diagnostics — sanity-check the batch as a whole

These charts are the practical version of the "how do I know it worked" question from the top of the notebook: they don't prove any single prediction is right, but they'll immediately surface a preprocessing mismatch or a nonsensical batch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

valid = results[results["pred_class"].notna()].copy()
valid["pred_class"] = valid["pred_class"].astype(int)

pred_counts = valid["pred_class"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)
pred_pct = 100 * pred_counts / max(len(valid), 1)
base_pct = 100 * np.array(TRAIN_BASE_COUNTS) / sum(TRAIN_BASE_COUNTS)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(NUM_CLASSES)
width = 0.38

BLUE = "#4C78A8"   # this batch
GRAY = "#B0B0B0"   # training baseline (reference, not the headline series)

ax.bar(x - width/2, base_pct, width, label="Training set (reference)", color=GRAY)
ax.bar(x + width/2, pred_pct.values, width, label="This batch (predicted)", color=BLUE)

ax.set_xticks(x)
ax.set_xticklabels(SHORT_NAMES, rotation=30, ha="right")
ax.set_ylabel("% of samples")
ax.set_title("Predicted class mix vs. training-set base rates")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"class_mix_{RUN_ID}.png", dpi=150)
plt.show()

print("If this batch's bars are wildly different in shape from the training reference")
print("(e.g. one class taking most of the mass), treat that as a preprocessing-parity")
print("red flag before treating it as a real finding about the facility.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(valid["confidence"].dropna(), bins=30, color=BLUE, edgecolor="white")
ax.axvline(CONFIDENCE_THRESHOLD, color="#B23A48", linestyle="--", linewidth=1.5,
           label=f"review threshold ({CONFIDENCE_THRESHOLD:.2f})")
ax.set_xlabel("confidence (max softmax probability)")
ax.set_ylabel("number of samples")
ax.set_title("Prediction confidence distribution")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"confidence_hist_{RUN_ID}.png", dpi=150)
plt.show()

median_conf = valid["confidence"].median()
print(f"Median confidence: {median_conf:.3f}")
print(f"Review queue: {n_review}/{len(results)} ({100*n_review/len(results):.1f}%)")
if median_conf < 0.5:
    print("\n⚠ Median confidence is very low across the whole batch — check preprocessing")
    print("  parity (crop coordinates, image format, sample rate) before trusting any output.")

## 9. Manual spot-check

Look at ~20-30 predictions yourself before handing anything to a reviewer: a few from the review queue (lowest confidence), a few random high-confidence ones, and specifically a few predicted as class 1 or 7. You already know what each class's spectrogram looks like from `Dataset_exploration.ipynb` — does the assigned label look plausible against the actual image?

In [ ]:
print("Lowest-confidence predictions (top of the review queue):")
display(results[results["needs_review"]].head(15)[
    ["filename", "pred_name", "confidence", "top2_class", "top2_prob"]
])

print("\nRandom sample of confident predictions:")
confident = results[~results["needs_review"]]
display(confident.sample(min(15, len(confident)), random_state=42)[
    ["filename", "pred_name", "confidence"]
])

## 10. What happens next (the feedback loop)

1. **Hand the review queue to a domain expert** (whoever normally labels these recordings) — not to re-run the whole pipeline, just to confirm/correct the flagged subset.
2. **Feed corrections back into the dataset.** Corrected class-1 (Hammering) and class-7 (Works/sirens) rows are exactly the extra rare-class data the README flags as the biggest remaining lever — append them to `Data/Annotations` and retrain, rather than letting them sit in a spreadsheet.
3. **Keep predictions traceable.** Each output CSV is stamped with `RUN_ID` and points at `CHECKPOINT_PATH` — if the model gets retrained later, old predictions should stay attributable to the checkpoint that made them.
4. **Watch for drift over time.** If you run this notebook periodically on new batches, keep the `class_mix_*.png` charts around and compare them run-to-run — a gradual shift can mean the facility's actual behavior is changing, or that something upstream (recording gear, MATLAB render settings) changed instead.
5. **Don't over-trust a single run.** One flipped sample moves rare-class numbers a lot (the README notes this for the labeled test set too) — treat any single batch's rare-class predictions as leads for a human to check, not conclusions.